# Physics-weight study of the PhysicsNeMo physics-informed Darcy FNO on a Kaggle GPU

Launcher for [physicsnemo-pino-darcy](https://github.com/AdebanjiAdelowo/physicsnemo-pino-darcy). It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on (the dataset is downloaded from Google Drive, about 0.8 GB).

| Step | What runs | When |
|---|---|---|
| A | dataset download and data check (`scripts/verify_data.py`) | always |
| | tests, CUDA smoke study, one-epoch timing probe | always |
| B | main study, `configs/full.yaml`: 27 runs of 50 epochs | `RUN_FULL = True` |

The upstream default configuration is the run `shipped_w0.1_seed0` of step B, so there is no separate control step. Before B the notebook projects the runtime from the probe and compares it with what is left of `SESSION_BUDGET_HOURS`. If the projection does not fit, the step is **not started and nothing is reduced**; the notebook reports the numbers and finishes.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-pino-darcy.git"
REF = "SET_TO_FULL_40_CHARACTER_COMMIT_SHA"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step B: only the value True starts it
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-pino-darcy"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

The requirements pin PhysicsNeMo to a commit of its main branch. `pip` may replace the PyTorch build of the image if it is older than PhysicsNeMo requires; the next cell prints the versions afterwards, in a new process, and checks CUDA again.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh('python -c "import torch, physicsnemo, warp, sympy; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, '| warp', warp.__version__, '| sympy', sympy.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)

## Step A: dataset and data check

Downloads `Darcy_241.zip`, writes the training, validation and test splits with their checksums, and checks what the data represent: grid, forcing, and the residual of the reference pressure under both residual definitions. The check runs on CPU and writes outside the clone.

In [ ]:
import json, zipfile
sh("python scripts/prepare_data.py", cwd=REPO)
CHECK = f"{WORK}/data-check"   # outside the clone, so the tree stays clean for the study
sh(f"python scripts/verify_data.py --out-root {CHECK}", cwd=REPO)
check = json.load(open(f"{CHECK}/results/data_check.json"))
local = json.load(open(f"{REPO}/results/data_check.json"))
print("archive sha256:", check["dataset"]["archive_sha256"])
print("DATASET IDENTICAL TO THE TRACKED CHECK:", check["dataset"]["files"] == local["dataset"]["files"])
with zipfile.ZipFile(f"{WORK}/physicsnemo-pino-darcy-data-check.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for rel in ("results/data_check.json", "figures/data_check.png"):
        z.write(f"{CHECK}/{rel}", rel)
assert tree_is_clean(), "step A modified the clone"

In [ ]:
sh("python -m pytest -q", cwd=REPO)

## CUDA smoke study

Three tiny models, two epochs on eight samples. The numbers are not results.

In [ ]:
sh("python scripts/train.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Timing probe for step B

Trains the model of the full study for one epoch, once without and once with the PDE loss. The projection extrapolates these two epochs to 50 epochs and 27 runs and adds nothing for evaluation.

In [ ]:
sh('python scripts/train.py --config full --no-evaluate name=full_probe max_epochs=1 "study.seeds=[0]" '
   '"study.residuals=[consistent]" "study.physics_weights=[0,0.1]"', cwd=REPO)
t_data = json.load(open(f"{REPO}/runs/full_probe/dataonly_seed0/train_metrics.json"))["train_seconds"]
t_pde = json.load(open(f"{REPO}/runs/full_probe/consistent_w0.1_seed0/train_metrics.json"))["train_seconds"]
n_data_only, n_physics = 3, 24  # configs/full.yaml: 3 seeds x (1 data-only + 2 residuals x 4 weights)
full_projection_h = (n_data_only * t_data + n_physics * t_pde) * 50 / 3600  # 50 epochs in configs/full.yaml
print(f"one epoch: {t_data:.1f} s data-only, {t_pde:.1f} s with the PDE loss, on {torch.cuda.get_device_name(0)}")
print(f"PROJECTION for step B: {full_projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")

## Step B: main study (`configs/full.yaml`)

Every run keeps its own directory with history, metrics and final checkpoint. `--resume` skips runs that already finished in this session.

In [ ]:
full_done = False
if RUN_FULL is not True:
    print("Step B skipped: RUN_FULL is False.")
elif full_projection_h > hours_left():
    print(f"STEP B NOT STARTED: projected {full_projection_h:.1f} h, {hours_left():.1f} h left of SESSION_BUDGET_HOURS = {SESSION_BUDGET_HOURS}.")
    print("The configuration was not reduced. Decide how to proceed before changing configs/full.yaml.")
else:
    sh("python scripts/train.py --config full --resume", cwd=REPO)
    sh("python scripts/plot_results.py --study runs/full --out runs/full/figures", cwd=REPO)
    sh("python scripts/results.py package runs/full", cwd=REPO)
    sh("python scripts/results.py bundle runs/full", cwd=REPO)
    sh(f"cp runs/full.zip {WORK}/physicsnemo-pino-darcy-full.zip && cp runs/full_eval_bundle.zip {WORK}/physicsnemo-pino-darcy-full-eval-bundle.zip", cwd=REPO)
    full_done = True
    print("STEP B COMPLETE.")

In [ ]:
import glob
print("Archives in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-pino-darcy-*.zip")):
    print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. Step A: done. Step B: {'done' if full_done else 'not run'}.")
if not full_done:
    print("KAGGLE CUDA CHECKS COMPLETE. No full study was run.")